## Web Scraping Amazon Data for Smartwatches

Note : Multi-Page Scraper with Error Protection will  run for several minutes. 

### Importing Essential Libraries

In [16]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

### Setting Scraping Parameters

In [17]:
BASE_URL = "https://www.amazon.in/s?k=smartwatch&rh=n%3A5605728031"

# Standard headers
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/98.0.4758.102 Safari/537.36',
    'Accept-Language': 'en-US, en;q=0.5'
}

# SETTING UP FOR THE BIG SCRAPE
PAGES_TO_SCRAPE = 400

# This list will hold all our data
products_data = [] 


This snippet establishes the parameters for an automated crawl of Amazon India's smartwatch category. By defining a specific BASE_URL, spoofing a browser via HEADERS, and setting a high PAGES_TO_SCRAPE limit, the code prepares the environment to extract and store large-scale product data into the products_data list.


### The Parsing Engine (Extracting the Data)

In [18]:
def parse_page_products(html_content):
    soup = BeautifulSoup(html_content, "html.parser")
    containers = soup.find_all('div', {'data-component-type': 's-search-result'})
    page_products = []

    for container in containers:
        product = {}

        # 1. Get Product Title
        try:
            product['title'] = container.h2.text.strip()
        except (AttributeError, TypeError):
            product['title'] = None

        # 2. Get Product Price
        try:
            price_element = container.find('span', class_='a-price-whole')
            product['price'] = price_element.text.strip().replace(',', '')
        except (AttributeError, TypeError):
            product['price'] = None

        # 3. Get Star Rating
        try:
            rating_span = container.find('span', class_='a-icon-alt')
            if rating_span and 'out of 5 stars' in rating_span.text:
                 product['rating'] = float(rating_span.text.split()[0])
            else:
                 product['rating'] = None
        except (AttributeError, ValueError, TypeError):
            product['rating'] = None
            
        # 4. Get Number of Reviews
        try:
            reviews_element = container.find('span', class_='a-size-mini puis-normal-weight-text s-underline-text')
            if reviews_element:
                product['reviews_count'] = reviews_element.text.strip().replace(',', '').replace('(', '').replace(')', '')
            else:
                product['reviews_count'] = None
        except (AttributeError, TypeError):
            product['reviews_count'] = None

        # 5. Get Product URL
        try:
            url_element = container.find('a', class_='a-link-normal s-no-outline')
            product['url'] = "https://www.amazon.in" + url_element['href']
        except (AttributeError, KeyError, TypeError):
            product['url'] = None
            
        # 6. Get Free Delivery Status
        try:
            delivery_element = container.find('div', class_='udm-primary-delivery-message')
            if delivery_element and 'FREE' in delivery_element.text.upper():
                product['free_delivery'] = True
            else:
                product['free_delivery'] = False
        except (AttributeError, TypeError):
            product['free_delivery'] = False

        # 7. Get Deal Badge
        try:
            deal_badge_element = container.find('span', class_='a-badge-text')
            if deal_badge_element:
                product['deal_badge'] = deal_badge_element.text.strip()
            else:
                product['deal_badge'] = None
        except (AttributeError, TypeError):
            product['deal_badge'] = None

        if product['title']:
            page_products.append(product)
        
    return page_products

This function defines the parsing logic to transform raw HTML into a structured list of dictionaries. It targets specific Amazon search result containers to extract key details including title, price, ratings, review counts, URLs, delivery status, and deals. While using "try-except" blocks to handle missing information gracefully, ensuring a clean dataset for every product found on the page.

### Multi-Page Scraper with Error Protection
* #### This cell will run for several minutes.

In [19]:
for page_num in range(1, PAGES_TO_SCRAPE + 1):
    url = f"{BASE_URL}&page={page_num}"
    

    try:
        response = requests.get(url, headers=HEADERS, timeout=15) # Increased timeout
        
        if response.status_code == 200:
            parsed_products = parse_page_products(response.content)
            products_data.extend(parsed_products)
            
        else:
            print(f"  > Failed to download page {page_num}. Status code: {response.status_code}. Stopping run.")
            break # Stop the loop if a page fails

    except requests.exceptions.RequestException as e:
        print(f"  > An error occurred: {e}. Stopping run.")
        break # Stop the loop if there's a network error
        
    # Wait for 3 seconds before the next request.
    time.sleep(3)

print(f"Finished scraping. Total raw products found: {len(products_data)}")


Finished scraping. Total raw products found: 10593


This loop executes the pagination logic by iterating through the specified number of search result pages. It constructs unique URLs for each page, performs the HTTP requests with error handling, and uses the previously defined parsing function to aggregate all extracted product details into a single master list. It also includes a politeness delay (time.sleep) to prevent being flagged as a bot.

### Converting Scraped Data to a DataFrame

In [20]:
# Convert the list of dictionaries into a pandas DataFrame
df = pd.DataFrame(products_data)

This line transforms the list of product dictionaries into a structured pandas DataFrame. It aligns each dictionary key as a column (e.g., Title, Price) and each product as a row, converting the raw scraped data into a clean table ready for analysis, filtering, or export.


### Data Quality Check and Inspection

In [21]:
# Display the first 10 rows of the DataFrame
df.head()

,title,price,rating,reviews_count,url,free_delivery,deal_badge
0,Fire-Boltt Ninja Call Pro Bluetooth Calling Sm...,999,3.9,1.2L,https://www.amazon.in/Fire-Boltt-Bluetooth-Cal...,True,None
1,Fire-Boltt Ninja Call Pro Plus Bluetooth Calli...,1099,3.9,1.2L,https://www.amazon.in/Fire-Boltt-Bluetooth-Cal...,True,None
2,"Fastrack Limitless Glide X 1.83"" Smart Watch w...",1299,4.0,3.6K,https://www.amazon.in/Fastrack-Limitless-Singl...,True,None
3,Fire-Boltt Ninja Call Pro Plus Bluetooth Calli...,1099,3.9,1.2L,https://www.amazon.in/Fire-Boltt-Bluetooth-Cal...,True,None
4,Boat Wave Call 3 Smartwatch 1.83” HD Display w...,1399,4.1,25.6K,https://www.amazon.in/Smartwatch-Display-Anima...,True,Amazon's


In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10593 entries, 0 to 10592
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   title          10593 non-null  object 
 1   price          10492 non-null  object 
 2   rating         5848 non-null   float64
 3   reviews_count  5848 non-null   object 
 4   url            10593 non-null  object 
 5   free_delivery  10593 non-null  bool   
 6   deal_badge     901 non-null    object 
dtypes: bool(1), float64(1), object(5)
memory usage: 507.0+ KB


In [23]:
df[df['title'].str.contains('Boat',na= False)].head(10)

,title,price,rating,reviews_count,url,free_delivery,deal_badge
4,Boat Wave Call 3 Smartwatch 1.83” HD Display w...,1399,4.1,25.6K,https://www.amazon.in/Smartwatch-Display-Anima...,True,Amazon's
15,Boat Wave Call 3 Smartwatch 1.83” HD Display w...,1399,4.1,25.6K,https://www.amazon.in/Smartwatch-Display-Anima...,True,None
39,Boat Wave Sigma 3 Curv Smartwatch with 2.01” (...,1499,5.0,3,https://www.amazon.in/Smartwatch-Immersive-Met...,True,None
44,"Boat Lunar Discovery w/ 1.39"" (3.5 cm) HD Disp...",1499,3.8,5.6K,https://www.amazon.in/boAt-Lunar-Discovery-Nav...,True,None
53,"Boat Wave Call Smart Watch for Men & Women, Bl...",1349,3.9,28.4K,https://www.amazon.in/boAt-Bluetooth-Calling-D...,True,None
55,Boat Ultima Ember smartwatch with 1.96” AMOLED...,2199,4.0,2.3K,https://www.amazon.in/boAt-Ultima-Ember-Smartw...,True,None
60,"Boat Chrome Horizon, Premium Metal Body, Video...",3599,4.0,444,https://www.amazon.in/boAt-Auto-Activity-Detec...,True,None
63,"Boat Chrome Horizon, Premium Metal Body, Video...",3599,4.0,444,https://www.amazon.in/sspa/click?ie=UTF8&spc=M...,True,None
68,Boat Wave Call 3 Smartwatch 1.83” HD Display w...,1399,4.1,25.6K,https://www.amazon.in/Wave-Smartwatch-Animated...,True,None
70,Boat Wave Call 3 Smartwatch 1.83” HD Display w...,1599,4.1,25.6K,https://www.amazon.in/Smartwatch-Animated-Func...,True,None


In [24]:
df['deal_badge'].value_counts()

deal_badge
Limited time deal    900
Amazon's               1
Name: count, dtype: int64

In [25]:
df['free_delivery'].value_counts()

free_delivery
False    5830
True     4763
Name: count, dtype: int64

The inspection confirms that the extracted data is accurate and well-structured. By using .head() to preview records, .info() to check for missing values, and value_counts() to verify data distribution, we ensured that the fields (like price and rating) are consistent and the scraper successfully captured the intended details from Amazon.


### Exporting Raw Data to CSV

In [26]:
# Save the DataFrame to a CSV file
df.to_csv('amazon_smartwatches_raw_data.csv', index=False)

print(f"Number of records saved sucessfully: {len(df)}")

Number of records saved sucessfully: 10593


This step permanently saves the scraped dataset into a CSV file named amazon_smartwatches_raw_data.csv. By setting index=False, it ensures a clean file structure without unnecessary row numbers, while the final print statement provides a success confirmation of the total number of records successfully archived.
